# Lecture 04 — Regular Expressions (Regex)

PGR215 Data Collection and Analysis — Kristiania University College

## 1. Hva er Regular Expressions?

**Regex** = et mønster-språk for å søke, matche og manipulere tekst.

I Python bruker vi `re`-modulen.

### Viktige re-funksjoner

| Funksjon | Beskrivelse | Returnerer |
|----------|------------|------------|
| `re.match(pattern, str)` | Matcher fra **starten** av strengen | Match-objekt eller None |
| `re.search(pattern, str)` | Finner **første** match hvor som helst | Match-objekt eller None |
| `re.findall(pattern, str)` | Finner **alle** matcher | Liste med strenger |
| `re.sub(pattern, repl, str)` | **Erstatter** alle matcher | Ny streng |

In [ ]:
import re

tekst = "Min e-post er anna@test.no og telefon er 91234567"

# match: sjekker bare starten
print("=== re.match ===")
result = re.match(r"Min", tekst)
print(f"Match 'Min': {result.group() if result else None}")
result = re.match(r"telefon", tekst)
print(f"Match 'telefon': {result}")  # None — matcher bare fra start

# search: finner første match hvor som helst
print("\n=== re.search ===")
result = re.search(r"telefon", tekst)
print(f"Search 'telefon': {result.group() if result else None}")

# findall: finner alle matcher
print("\n=== re.findall ===")
tall = re.findall(r"\d+", tekst)
print(f"Alle tall: {tall}")

# sub: erstatt
print("\n=== re.sub ===")
censored = re.sub(r"\d", "X", tekst)
print(f"Sensurert: {censored}")

## 2. Spesialtegn i Regex

### Tegnklasser

| Mønster | Matcher | Eksempel |
|---------|---------|----------|
| `.` | Ethvert tegn (unntatt newline) | `a.c` → "abc", "a1c" |
| `\d` | Siffer (0-9) | `\d\d` → "42" |
| `\D` | Ikke-siffer | `\D+` → "abc" |
| `\w` | Ord-tegn (a-z, A-Z, 0-9, _) | `\w+` → "hello_1" |
| `\W` | Ikke ord-tegn | `\W` → "@", " " |
| `\s` | Whitespace (mellomrom, tab, newline) | `\s+` → "  " |
| `\S` | Ikke-whitespace | `\S+` → "hello" |

### Kvantifiserere

| Mønster | Betydning | Eksempel |
|---------|-----------|----------|
| `*` | 0 eller flere | `ab*c` → "ac", "abc", "abbc" |
| `+` | 1 eller flere | `ab+c` → "abc", "abbc" (ikke "ac") |
| `?` | 0 eller 1 | `colou?r` → "color", "colour" |
| `{n}` | Nøyaktig n | `\d{4}` → "2024" |
| `{n,m}` | Mellom n og m | `\d{2,4}` → "12", "123", "1234" |

### Ankere og grupper

| Mønster | Betydning |
|---------|-----------|
| `^` | Start av streng |
| `$` | Slutt av streng |
| `[abc]` | Tegnklasse: a, b, eller c |
| `[^abc]` | Negert: alt UNNTATT a, b, c |
| `(...)` | Gruppe (capture group) |
| `\|` | Eller |

In [ ]:
# Demonstrer spesialtegn
print("=== Tegnklasser ===")
tekst = "Ordre #42: 3 laptops à kr 12.500,-"
print(f"Tekst: {tekst}")
print(f"\\d+ (siffer):   {re.findall(r'\d+', tekst)}")
print(f"\\D+ (ikke-siffer): {re.findall(r'\D+', tekst)}")
print(f"\\w+ (ord):      {re.findall(r'\w+', tekst)}")

print("\n=== Kvantifiserere ===")
print(f"\\d{{2}}:    {re.findall(r'\d{2}', '1 22 333 4444')}")  
print(f"\\d{{2,3}}: {re.findall(r'\d{2,3}', '1 22 333 4444')}")
print(f"\\d+:     {re.findall(r'\d+', '1 22 333 4444')}")

print("\n=== Tegnklasser [] ===")
print(f"[aeiou]:  {re.findall(r'[aeiou]', 'Hello World')}")
print(f"[A-Z]:    {re.findall(r'[A-Z]', 'Hello World')}")
print(f"[0-9]+:   {re.findall(r'[0-9]+', 'Tel: 912-34-567')}")

## 3. Praktisk: Trekke ut e-poster og telefonnumre

In [ ]:
# Ekstraher e-postadresser
tekst = """
Kontakt oss:
- Support: support@firma.no
- Salg: salg.avd@firma.com  
- CEO: ole.hansen@kristiania.no
- Ugyldig: @mangler.no
- Også ugyldig: test@
"""

epost_pattern = r'[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}'
eposter = re.findall(epost_pattern, tekst)
print("Gyldige e-poster funnet:")
for e in eposter:
    print(f"  {e}")

In [ ]:
# Ekstraher norske telefonnumre
tekst = """
Ring oss: 22 33 44 55 (kontor)
Mobil: 912 34 567
Annet: +47 48765432
For kort: 1234
Utenlandsk: +1-555-0123
"""

# Norske numre: 8 siffer, med eller uten +47, med eller uten mellomrom
telefon_pattern = r'(?:\+47\s?)?\d{2,3}\s?\d{2,3}\s?\d{2,3}'
telefoner = re.findall(telefon_pattern, tekst)
print("Telefonnumre funnet:")
for t in telefoner:
    # Rens: fjern mellomrom
    clean = re.sub(r'\s', '', t)
    print(f"  {t!r:25} → renset: {clean}")

## 4. Regex for Data Cleaning

In [ ]:
import pandas as pd

# Rens uryddig data med regex
df = pd.DataFrame({
    'navn': ['  Anna Berg ', 'erik HANSEN', 'Sara  Olsen', 'OLE lie'],
    'telefon': ['912 34 567', '+47-22334455', '(47) 987 65 432', 'tlf: 41234567'],
    'epost': ['anna@test.no', 'ERIK@Test.NO', 'sara@@test.no', 'ole@test'],
    'belop': ['kr 1.500,-', 'NOK 2500', '1,200.50', '3000 kr']
})
print("Før rensing:")
display(df)

# Rens telefonnumre: behold bare siffer
df['telefon_clean'] = df['telefon'].apply(lambda x: re.sub(r'\D', '', x))
# Fjern eventuell landkode 47 fra starten
df['telefon_clean'] = df['telefon_clean'].apply(
    lambda x: x[2:] if x.startswith('47') and len(x) == 10 else x
)

# Rens beløp: trekk ut tall
df['belop_clean'] = df['belop'].apply(
    lambda x: re.sub(r'[^\d.]', '', x.replace(',', ''))
).astype(float)

# Valider e-post
df['epost_gyldig'] = df['epost'].apply(
    lambda x: bool(re.match(r'^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$', x))
)

print("\nEtter rensing:")
display(df)

## 5. Capture Groups — Trekke ut deler av en match

In [ ]:
# Bruk grupper () for å trekke ut deler
tekst = "Ordredato: 2024-03-15, Leveringsdato: 2024-03-20"

# Trekk ut datoer og del opp i år, måned, dag
pattern = r'(\d{4})-(\d{2})-(\d{2})'
for match in re.finditer(pattern, tekst):
    print(f"Full match: {match.group(0)}")
    print(f"  År:   {match.group(1)}")
    print(f"  Måned: {match.group(2)}")
    print(f"  Dag:  {match.group(3)}")
    print()

# Named groups
pattern_named = r'(?P<aar>\d{4})-(?P<maned>\d{2})-(?P<dag>\d{2})'
match = re.search(pattern_named, tekst)
if match:
    print(f"Named groups: {match.groupdict()}")

## 6. re.sub — Søk og erstatt

In [ ]:
# Erstatt med regex
print("=== Anonymisering ===")
tekst = "Anna (fnr: 01019012345) tjener 750000 kr"
anonymisert = re.sub(r'\d{11}', 'XXXXX-XXXXX', tekst)
anonymisert = re.sub(r'\d{6}', 'XXX.XXX', anonymisert)
print(f"Original:    {tekst}")
print(f"Anonymisert: {anonymisert}")

print("\n=== Fjern HTML-tags ===")
html = "<h1>Tittel</h1><p>Dette er <b>viktig</b> tekst.</p>"
ren_tekst = re.sub(r'<[^>]+>', '', html)
print(f"HTML:  {html}")
print(f"Tekst: {ren_tekst}")

print("\n=== Standardiser datoformat ===")
datoer = ["15/03/2024", "20-04-2024", "10.05.2024"]
for d in datoer:
    standard = re.sub(r'(\d{2})[/.-](\d{2})[/.-](\d{4})', r'\3-\2-\1', d)
    print(f"  {d} → {standard}")

## 7. Regex Cheat Sheet

```
Tegn:     .  \d  \w  \s  \D  \W  \S
Mengde:   *  +  ?  {n}  {n,m}
Ankere:   ^  $  \b
Grupper:  (...)  (?:...)  (?P<name>...)
Klasser:  [abc]  [a-z]  [^abc]
Escape:   \. \* \+ \? \( \)
Flagg:    re.IGNORECASE  re.MULTILINE  re.DOTALL
```

In [ ]:
# Quiz: Test din regex-forståelse
test_cases = [
    (r'\d{3}-\d{4}', "Ring 123-4567 nå!", "Telefon (XXX-XXXX)"),
    (r'[A-Z]{2}\d{4}', "Regnr: AB1234 og CD5678", "Norske bilskilt"),
    (r'\b\w{5}\b', "Hei på deg min venn", "Ord med nøyaktig 5 bokstaver"),
    (r'https?://\S+', "Besøk https://kristiania.no i dag", "URL-er"),
]

for pattern, tekst, beskrivelse in test_cases:
    result = re.findall(pattern, tekst)
    print(f"{beskrivelse}")
    print(f"  Pattern: {pattern}")
    print(f"  Tekst:   {tekst}")
    print(f"  Resultat: {result}\n")

## 8. Regex i Pandas

In [ ]:
# Bruk regex direkte i pandas
df = pd.DataFrame({
    'tekst': [
        'Ordre #123 fra anna@test.no',
        'Ordre #456 fra erik@firma.com',
        'Faktura #789 til sara@skole.no',
        'Ordre #012 fra ole@test.no'
    ]
})

# str.extract — trekk ut med capture groups
df['nummer'] = df['tekst'].str.extract(r'#(\d+)')
df['epost'] = df['tekst'].str.extract(r'([\w.]+@[\w.]+)')
df['domene'] = df['epost'].str.extract(r'@(\w+)\.\w+')

# str.contains — filtrer rader
df['er_ordre'] = df['tekst'].str.contains(r'Ordre', regex=True)

# str.replace — regex-erstatt
df['anonymisert'] = df['tekst'].str.replace(r'[\w.]+@[\w.]+', '***@***.***', regex=True)

display(df)

## Oppsummering

**Nøkkelkonsepter fra Lecture 04:**

1. `re.match()` — matcher fra starten
2. `re.search()` — finner første match
3. `re.findall()` — finner alle matcher
4. `re.sub()` — søk og erstatt
5. `\d`, `\w`, `\s` for siffer, ord-tegn, whitespace
6. `*`, `+`, `?`, `{n}` for mengde
7. `()` for capture groups
8. Pandas: `.str.extract()`, `.str.contains()`, `.str.replace()`